# 🖥️ MITgcm tutorial

This tutorial will show how to run a 3D simulation with output from the MITgcm model. The key thing about MITgcm is that its output is on a C-grid, which means that the U and V velocity components are not defined at the same location as the particle positions. Parcels has built-in functionality to deal with C-grid data. See also the [Nemo tutorial](tutorial_nemo.ipynb) for another example of C-grid data.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import parcels
import parcels.tutorial

ds_fields = parcels.tutorial.open_dataset(
    "MITgcm_example_data/mitgcm_UV_surface_zonally_reentrant"
)

We can use a combination of {py:func}`parcels.convert.mitgcm_to_sgrid` and {py:func}`parcels.FieldSet.from_sgrid_conventions` to read in the data. See below for an example.

In [ ]:
ds_fset = parcels.convert.mitgcm_to_sgrid(
    fields={"U": ds_fields.UVEL, "V": ds_fields.VVEL}, coords=ds_fields
)
fieldset = parcels.FieldSet.from_sgrid_conventions(ds_fset)

We can then run a simulation on this zonally re-entrant dataset as follows:

In [ ]:
perc = np.arange(5, 100, 10)
lon_vals = np.percentile(fieldset.UV.grid.lon, perc)
lat_vals = np.percentile(fieldset.UV.grid.lat, perc)

X, Y = np.meshgrid(
    lon_vals,
    lat_vals,
)
Z = np.zeros_like(X)


def DeleteParticle(particles, fieldset):
    any_error = particles.state >= 50
    particles[any_error].state = parcels.StatusCode.Delete


pset = parcels.ParticleSet(fieldset=fieldset, x=X, y=Y, z=Z)

outputfile = parcels.ParticleFile(
    path="mitgcm_particles.parquet",
    outputdt=np.timedelta64(5000, "s"),
)

pset.execute(
    [parcels.kernels.AdvectionRK2, DeleteParticle],
    runtime=np.timedelta64(10, "D"),
    dt=np.timedelta64(1, "h"),
    output_file=outputfile,
)

And make a simple plot of the particle trajectories:

In [ ]:
df = parcels.read_particlefile("mitgcm_particles.parquet")

for traj in df.partition_by("particle_id", maintain_order=True):
    plt.plot(traj["x"], traj["y"], ".-")
plt.show()